# Exploring a graph: the twelve questions

Twelve questions a user asks while exploring, on one deterministic tiny
graph: two aspects (ordered `time`, categorical `condition`), a node
repeated on several layers, isolated nodes, directed and undirected edges,
directed and undirected hyperedges, a cross-layer coupling edge, a self-loop, a
half-edge, an edge whose endpoint is another edge, two overlapping slices and
attributes at all eight addresses — with nulls, a field named with `__`, and
labels that collide under naive underscore joining. No download, no biology.

Every cell answers a question a user asks while exploring, with no helper
function: no coordinate indexing by hand, no set intersection of ids, no
hyperedge string parsing, no cache refresh and no copying of metadata. The
same scenarios are pinned in `tests/test_exploration_workflow.py`.

In [1]:
import warnings
import numpy as np
from annnet import AnnNet

warnings.simplefilter('ignore')  # placeholder-layer notices from the edge entity

G = AnnNet(directed=True, aspects={'time': ['0h', '6h', '24h'], 'condition': ['ctrl', 'stim', 'stim_x']})
G.layers.set_ordered('time')
G.layers.place(['a'], [('0h', 'stim'), ('6h', 'stim'), ('24h', 'stim')])
G.layers.place(['b'], [('0h', 'ctrl'), ('6h', 'ctrl'), ('0h', 'stim'), ('6h', 'stim')])
G.layers.place(['c', 'iso'], [('0h', 'stim')])
G.add_nodes('lonely', layer=('24h', 'stim_x'))
G.add_edges(('a', ('0h', 'stim')), ('b', ('0h', 'stim')), edge_id='ab0', weight=0.9)
G.add_edges(('a', ('6h', 'stim')), ('b', ('6h', 'stim')), edge_id='ab6', weight=0.4)
G.add_edges(('b', ('0h', 'ctrl')), ('b', ('0h', 'stim')), edge_id='b_cross', weight=0.2)   # coupling
G.add_edges(('b', ('0h', 'stim')), ('c', ('0h', 'stim')), edge_id='bc', directed=False)
G.add_edges(('c', ('0h', 'stim')), ('c', ('0h', 'stim')), edge_id='loop')
G.add_edges([
    {'head': [('a', ('0h', 'stim'))], 'tail': [('b', ('0h', 'stim')), ('c', ('0h', 'stim'))], 'edge_id': 'h_dir'},
    {'members': [('a', ('6h', 'stim')), ('b', ('6h', 'stim'))], 'edge_id': 'h_undir'},
    {'head': [('c', ('0h', 'stim'))], 'tail': [], 'edge_id': 'half'},
])
G.add_edges(('a', ('24h', 'stim')), ('b', ('6h', 'stim')), edge_id='backing', as_entity=True)
G.add_edges('backing', 'c', edge_id='meta')   # an edge whose endpoint is an edge
G.slices.add('prior', edges=['ab0', 'ab6', 'bc', 'h_dir'])
G.slices.add('fit', edges=['ab0', 'h_dir', 'h_undir', 'meta'])
G.attrs.update('nodes', {'a': {'group': 'x', 'score': 0.9, 'weird__name': 1}, 'b': {'group': 'x', 'score': 0.3, 'weird__name': 2},
                         'c': {'group': 'y', 'score': None}, 'iso': {'group': 'y', 'score': 0.7}})
G.attrs.update('edges', {'ab0': {'confidence': 0.95, 'source_db': 'A'}, 'ab6': {'confidence': 0.5}, 'bc': {'confidence': 0.8},
                         'h_dir': {'confidence': 0.9}, 'h_undir': {'confidence': 0.85}, 'meta': {'confidence': 0.7}})
G.attrs.update('slices', {'prior': {'role': 'knowledge'}, 'fit': {'role': 'fitted'}})
G.attrs.update('aspects', {'time': {'unit': 'h'}})
G.attrs.update('layers', {('0h', 'stim'): {'n_cells': 100}, ('6h', 'stim'): {'n_cells': 80}})
G.attrs.update('edge_slices', {('fit', 'ab0'): {'activity': 1.2}, ('fit', 'h_dir'): {'activity': -0.5}, ('fit', 'meta'): {'activity': 0.1}})
G.attrs.update('node_layers', {('a', ('0h', 'stim')): {'expr': 0.95}, ('a', ('6h', 'stim')): {'expr': 0.2}, ('b', ('0h', 'stim')): {'expr': 0.85}})
G.attrs.update('elementary_layers', {('condition', 'stim_x'): {'note': 'collides'}, ('time', '0h'): {'note': 'baseline'}})
G.uns['study'] = 'demo'
G

AnnNet object with n_nodes × n_edges = 5 × 10
    directed: True
    slices: ['default', 'prior', 'fit']
    aspects: ['time', 'condition']
    supra_nodes (node × layer rows): 10
    attrs.nodes: ['group', 'score', 'weird__name']
    attrs.edges: ['confidence', 'source_db']
    uns: ['study']

## 1. Inspect the schema and the summary without building a table or a matrix

In [2]:
G.summary()

AnnNet summary: 5 node(s), 10 edge(s), 10 placement(s), 1 edge entity
  directed default: True
  aspects (2): time (ordered): [0h, 6h, 24h]; condition: [ctrl, stim, stim_x]
  layers occurring: 6
  slices (3): default (6n/10e), prior (3n/4e), fit (4n/4e)
  attrs: nodes: ['group', 'score', 'weird__name']; edges: ['confidence', 'source_db']; slices: ['role']; aspects: ['unit']; layers: ['n_cells']; edge_slices: ['activity']; node_layers: ['expr']; elementary_layers: ['note']
  edges (computed): binary=6, hyper=3, node_edge=1; directed=7, undirected=3

In [3]:
G.attrs.schema('edges', derived=True)

Schema('edges', derived, 10 row(s))
  keys:   edge_id: str
  fields: confidence: float, source_db: str, kind: str, directed: bool, weight: float, ml_kind: str, source: str, target: str, src_layer: str, dst_layer: str, head: list, tail: list, members: list, effective_weight: float

In [4]:
G.attrs.schema('elementary_layers')

Schema('elementary_layers', 6 row(s))
  keys:   aspect: str, elementary_layer: str
  fields: note: str

## 2. Threshold, category and null filters, composed with OR / AND / difference

In [5]:
print(G.N.select(score__gte=0.5).ids)
print(G.N.select(group='y', score__isnull=True).ids)
chosen = (G.N.select(group='x') | G.N.select(score__gt=0.5)) - G.N.select(weird__name=2)
print(chosen.ids)
print(G.E.select(confidence__gte=0.8, kind='hyper').ids)
print(G.E.select(kind='binary', directed=False).ids)

('a', 'iso')
('c',)
('a', 'iso')
('h_dir', 'h_undir')
('bc',)


## 3. Restrict to ordered layers and a slice, then to high-confidence directed hyperedges

In [6]:
early = G.layers.where(time__lte='6h')
V = G.view(layers=early, slices='fit')
print(V.E.ids)
W = V.view(edges=V.E.select(confidence__gte=0.85, kind='hyper', directed=True))
print(W.E.ids, W.supra_nodes())

('ab0', 'h_dir', 'h_undir')
('h_dir',) [('a', ('0h', 'stim')), ('b', ('0h', 'stim')), ('c', ('0h', 'stim'))]


## 4. Complete endpoints and their coordinates through the incidence layout — nothing parsed

In [7]:
table = G.attrs.table('edges', derived=True, layout='incidences', backend='pandas')
table[table['edge_id'].isin(['h_dir', 'meta', 'b_cross'])]

,edge_id,position,entity_id,entity_kind,layer,layer_id,role,coefficient,kind,directed
4,b_cross,0,b,node,"[0h, ctrl]",0h×ctrl,source,0.2,binary,True
5,b_cross,1,b,node,"[0h, stim]",0h×stim,target,-0.2,binary,True
10,h_dir,0,a,node,"[0h, stim]",0h×stim,source,1.0,hyper,True
11,h_dir,1,b,node,"[0h, stim]",0h×stim,target,-1.0,hyper,True
12,h_dir,2,c,node,"[0h, stim]",0h×stim,target,-1.0,hyper,True
18,meta,0,backing,edge,"[_, _]",_×_,source,1.0,binary,True
19,meta,1,c,node,"[0h, stim]",0h×stim,target,-1.0,binary,True


In [8]:
# Joinable against the node-layer address by (entity_id, layer_id): one row per endpoint, no parsing.
node_layers = G.attrs.table('node_layers', derived=True, backend='pandas')   # derived: adds the display layer_id
(table[table['edge_id'] == 'h_dir']
 .merge(node_layers, left_on=['entity_id', 'layer_id'], right_on=['node_id', 'layer_id'], how='left')
 [['edge_id', 'entity_id', 'layer_id', 'role', 'coefficient', 'expr']])

,edge_id,entity_id,layer_id,role,coefficient,expr
0,h_dir,a,0h×stim,source,1.0,0.95
1,h_dir,b,0h×stim,target,-1.0,0.85
2,h_dir,c,0h×stim,target,-1.0,NaN


## 5. Select a node by a contextual score without pulling in its other placements

In [9]:
rows = G.attrs.select('node_layers', expr__gte=0.8)
print(rows.keys)
V = G.view(nodes=rows)
print(V.supra_nodes(), V.E.ids)
print(rows.project('nodes').ids)

(('a', ('0h', 'stim')), ('b', ('0h', 'stim')))
[('a', ('0h', 'stim')), ('b', ('0h', 'stim'))] ('ab0',)
('a', 'b')


## 6. Positive edge-slice values, projected explicitly onto edges

In [10]:
fit = G.attrs.select('edge_slices', slice_id='fit', activity__gt=0)
print(fit.keys)
try:
    G.view(edges=fit)
except TypeError as err:
    print('refused:', err)
V = G.view(edges=fit.project('edges'))
print(V.E.ids, '<- meta brings its backing edge')
print(fit.project('slices').ids)

(('fit', 'ab0'), ('fit', 'meta'))
refused: edge_slices rows carry a slice: project them explicitly with .project('edges') (or .project('slices')) so the slice meaning is not silently discarded
('ab0', 'backing', 'meta') <- meta brings its backing edge
('fit',)


## 7. Compare two slices and read a numeric frame

In [11]:
G.slices.compare('prior', 'fit', backend='pandas')

,edge_id,status
0,ab0,both
1,ab6,a_only
2,bc,a_only
3,h_dir,both
4,h_undir,b_only
5,meta,b_only


In [12]:
both = G.view(slices=G.slices.intersect(['prior', 'fit']))
print(both.E.ids)
G.slices.edge_frame(edges=['ab0', 'h_dir'], slices=['fit'], attrs=['activity'], backend='pandas')

('ab0', 'h_dir')


,slice_id,ab0,h_dir
0,fit,1.2,-0.5


In [13]:
matrix = G.layers.matrix('expr', nodes=['a', 'b'], layers=[('0h', 'stim'), ('6h', 'stim')])
matrix.values

array([[0.95, 0.85],
       [0.2 ,  nan]])

## 8. Closed against open boundary, without truncating a hyperedge

In [14]:
closed = G.view(nodes=[('a', ('0h', 'stim'))])
opened = G.view(nodes=[('a', ('0h', 'stim'))], boundary='open')
print(closed.E.ids, opened.E.ids)
print(opened.supra_nodes())
print(opened.summary()['expanded'])
window = G.layers.where(time='0h', condition='stim')
print(window.crossing, window.boundary)

() ('ab0', 'h_dir')
[('a', ('0h', 'stim')), ('b', ('0h', 'stim')), ('c', ('0h', 'stim'))]
{'nodes': 2, 'edges': 0}
{'b_cross', 'meta'} {'c', 'b'}


## 9. A further filter on a view cannot broaden it

In [15]:
V = G.view(layers=G.layers.where(time='0h'))
W = V.view(layers=G.layers.where(time__gte='0h'), slices=['prior', 'fit', 'default'])
print(set(W.supra_nodes()) <= set(V.supra_nodes()), set(W.E.ids) <= set(V.E.ids))
print(V.view(nodes=list(G.N)).N.ids == V.N.ids, V.view(boundary='open').E.ids == V.E.ids)

True True
True True


## 10. Edit the parent and watch the live view change; every write through the view fails

In [16]:
V = G.view(nodes=G.N.select(score__gte=0.5))
print(V.N.ids)
G.attrs.update('nodes', {'b': {'score': 0.6}})
print(V.N.ids)
from annnet.core._select import ReadOnlyViewError
for attempt in (lambda: V.N.__setitem__('score', 1.0),
                lambda: V.attrs.update('nodes', {'a': {'score': 1.0}}),
                lambda: V.uns.__setitem__('x', 1)):
    try:
        attempt()
    except ReadOnlyViewError as err:
        print('refused:', str(err)[:60], '...')
try:
    V.add_nodes('z')
except AttributeError as err:
    print('absent:', err)

('a', 'iso')
('a', 'b', 'iso')
refused: a view is read-only; call .materialize() for an independent  ...
refused: a view is read-only; call .materialize() for an independent  ...
refused: the metadata of a view is read-only; call .materialize() for ...
absent: a view has no 'add_nodes': it is read-only. Call .materialize() for an independent graph, or apply the operation to the parent graph.


## 11. Materialize, edit the copy, and the parent is unchanged

In [17]:
V = G.view(layers=G.layers.where(time='0h'))
H = V.materialize()
H.add_nodes('new', layer=('0h', 'stim'))
H.attrs.update('nodes', {'a': {'group': 'changed'}})
H.remove_edges('ab0')
print('new' in G.N, G.attrs.row('nodes', 'a')['group'], 'ab0' in G.E, 'ab0' in V.E.ids)
H.uns['selection']

False x True True


{'nodes': 4,
 'supra_nodes': 5,
 'edges': 6,
 'layers': [['0h', 'ctrl'], ['0h', 'stim'], ['0h', 'stim_x']],
 'slices': ['default', 'prior', 'fit'],
 'boundary': 'closed',
 'filters': ['layers'],
 'expanded_nodes': [],
 'expanded_edges': [],
 'structure_version': 15}

## 12. Selected ids, all eight attribute addresses, matrix labels and an IO round trip agree

In [18]:
import tempfile
from pathlib import Path

V = G.view(layers=G.layers.where(time__lte='6h', condition='stim'), slices=['prior', 'fit'])
H = V.materialize()
print(list(H.N) == list(V.N.ids), list(H.E) == list(V.E.ids), list(H.supra_nodes()) == list(V.supra_nodes()))
for address in V.attrs:
    v_rows, h_rows = V.attrs.rows(address), H.attrs.rows(address)
    print(f'{address:18}', {k: h_rows[k] for k in v_rows} == v_rows)
print(V.S.shape == H.S.shape and np.array_equal(V.S.toarray(), H.S.toarray()))
print([V.idx.col_to_edge(j) for j in range(V.S.shape[1])] == list(V.E.ids))
with tempfile.TemporaryDirectory() as tmp:
    H.write(Path(tmp) / 'selected.annnet')
    back = AnnNet.read(Path(tmp) / 'selected.annnet')
print(list(back.N) == list(H.N), list(back.E) == list(H.E), np.array_equal(back.S.toarray(), H.S.toarray()))
print(back.layers.aspect('time').ordered, back.attrs.rows('elementary_layers') == H.attrs.rows('elementary_layers'))

True True True
nodes              True
edges              True
slices             True
aspects            True
layers             True
edge_slices        True
node_layers        True
elementary_layers  True
True
True


True True True
True True
